# Notebook 2 — YouCook2 Download & Annotation Parsing
Downloads the YouCook2 annotation JSON, parses it into a flat dataframe,
and downloads the raw YouTube videos using yt-dlp.

**Outputs:**
- `annotations/segments.csv` — all segments
- `annotations/segments_available.csv` — segments whose video downloaded successfully
- `annotations/download_report.csv` — per-video download status
- `videos/*.mp4` — raw video files


In [ ]:
import os, json, pandas as pd

with open('/home/ubuntu/youcook2_project/paths.json') as f:
    PATHS = json.load(f)

print("Paths loaded:")
for k, v in PATHS.items():
    print(f"  {k}: {v}")


## Download YouCook2 annotations
The tarball contains train + validation splits with segment boundaries and captions.
If the primary URL is down, the code automatically tries the GitHub mirror.


In [ ]:
import subprocess, os

ANN_DIR  = PATHS['annotations']
ANN_TAR  = os.path.join(ANN_DIR, 'youcookii_annotations_trainval.tar.gz')
ANN_JSON = os.path.join(ANN_DIR, 'youcookii_annotations_trainval.json')

if os.path.exists(ANN_JSON):
    print(f"Annotations already present ({os.path.getsize(ANN_JSON)/1e6:.1f} MB) ✓")
else:
    primary = "http://youcook2.eecs.umich.edu/static/YouCookII/youcookii_annotations_trainval.tar.gz"
    mirror  = "https://raw.githubusercontent.com/LuoweiZhou/YouCook2-Leaderboard/master/data/youcookii_annotations_trainval.json"

    print("Downloading annotations (primary)...")
    res = subprocess.run(['wget', '-q', '--show-progress', '-O', ANN_TAR, primary])

    if res.returncode == 0 and os.path.getsize(ANN_TAR) > 10_000:
        subprocess.run(['tar', '-xzf', ANN_TAR, '-C', ANN_DIR], check=True)
        # Locate and normalise the JSON filename
        for root, _, files in os.walk(ANN_DIR):
            for fn in files:
                if fn.endswith('.json') and 'annotation' in fn.lower():
                    src = os.path.join(root, fn)
                    if src != ANN_JSON:
                        os.rename(src, ANN_JSON)
                    break
        print("Extracted ✓")
    else:
        print("Primary URL failed — trying GitHub mirror...")
        subprocess.run(['wget', '-q', '--show-progress', '-O', ANN_JSON, mirror], check=True)

    print(f"Done. File size: {os.path.getsize(ANN_JSON)/1e6:.1f} MB")


## Parse JSON → flat dataframe
YouCook2 structure: `database → video_id → {subset, recipe_type, annotations: [{id, segment, sentence}]}`  
We flatten to one row per segment.


In [ ]:
import json, pandas as pd

with open(ANN_JSON) as f:
    raw = json.load(f)

database = raw.get('database', raw)

rows = []
for video_id, info in database.items():
    for ann in info.get('annotations', []):
        seg = ann['segment']
        rows.append({
            'video_id':    video_id,
            'subset':      info.get('subset', 'unknown'),
            'recipe_type': info.get('recipe_type', 'unknown'),
            'segment_id':  f"{video_id}_{ann['id']:03d}",
            'ann_id':      ann['id'],
            'start_sec':   float(seg[0]),
            'end_sec':     float(seg[1]),
            'duration':    float(seg[1]) - float(seg[0]),
            'caption':     ann['sentence'].strip(),
        })

df = pd.DataFrame(rows)
print(f"Total segments : {len(df):,}")
print(f"Unique videos  : {df['video_id'].nunique():,}")
print(f"Train segments : {(df['subset']=='training').sum():,}")
print(f"Val segments   : {(df['subset']=='validation').sum():,}")
print(f"Avg duration   : {df['duration'].mean():.1f}s")

csv_path = os.path.join(PATHS['annotations'], 'segments.csv')
df.to_csv(csv_path, index=False)
print(f"\nSaved → {csv_path}")
df.head(3)


## Download videos
YouCook2 videos are public YouTube uploads. We use yt-dlp at 360p to keep file sizes
manageable while preserving enough quality for frame extraction.

This cell is **resumable** — already-downloaded videos are skipped automatically.
Expect ~15–20% of videos to be unavailable (deleted/private over time).

On the A100 instance, network throughput is high so the full dataset should download in 2–4 hours.


In [ ]:
import subprocess, os
from tqdm.notebook import tqdm

VIDEO_DIR = PATHS['videos']
video_ids = df['video_id'].unique()
print(f"Unique videos to fetch: {len(video_ids)}")

YDL_BASE = [
    'yt-dlp',
    '--format', 'bestvideo[height<=360][ext=mp4]+bestaudio[ext=m4a]/best[height<=360][ext=mp4]/best[height<=360]',
    '--merge-output-format', 'mp4',
    '--no-playlist',
    '--quiet',
    '--no-warnings',
    '--socket-timeout', '30',
    '--retries', '3',
    '--concurrent-fragments', '4',
]

results = {'success': [], 'skipped': [], 'failed': []}

for vid_id in tqdm(video_ids, desc="Downloading videos"):
    out_path = os.path.join(VIDEO_DIR, f"{vid_id}.mp4")

    if os.path.exists(out_path) and os.path.getsize(out_path) > 50_000:
        results['skipped'].append(vid_id)
        continue

    output_tmpl = os.path.join(VIDEO_DIR, '%(id)s.mp4')
    url = f"https://www.youtube.com/watch?v={vid_id}"
    try:
        res = subprocess.run(
            YDL_BASE + ['--output', output_tmpl, url],
            capture_output=True, text=True, timeout=180
        )
        if res.returncode == 0 and os.path.exists(out_path):
            results['success'].append(vid_id)
        else:
            results['failed'].append(vid_id)
    except subprocess.TimeoutExpired:
        results['failed'].append(vid_id)

print(f"\nSuccess : {len(results['success']):,}")
print(f"Skipped : {len(results['skipped']):,}  (already on disk)")
print(f"Failed  : {len(results['failed']):,}  (deleted/private — normal)")


In [ ]:
# Save download report and filter segments to available videos
import pandas as pd

report = (
    [{'video_id': v, 'status': 'success'} for v in results['success']] +
    [{'video_id': v, 'status': 'skipped'} for v in results['skipped']] +
    [{'video_id': v, 'status': 'failed'}  for v in results['failed']]
)
pd.DataFrame(report).to_csv(
    os.path.join(PATHS['annotations'], 'download_report.csv'), index=False
)

available = set(results['success'] + results['skipped'])
df_avail  = df[df['video_id'].isin(available)].copy().reset_index(drop=True)
df_avail.to_csv(os.path.join(PATHS['annotations'], 'segments_available.csv'), index=False)

print(f"Available segments: {len(df_avail):,} / {len(df):,}")


## Inspect a sample segment
Quick sanity check — confirms decord can read the files we downloaded.


In [ ]:
from decord import VideoReader, cpu
import matplotlib.pyplot as plt

sample = df_avail.sample(1).iloc[0]
vid_path = os.path.join(PATHS['videos'], f"{sample['video_id']}.mp4")

print(f"Segment : {sample['segment_id']}")
print(f"Caption : {sample['caption']}")
print(f"Span    : {sample['start_sec']:.1f}s → {sample['end_sec']:.1f}s")

vr  = VideoReader(vid_path, ctx=cpu(0))
fps = vr.get_avg_fps()
mid = int(((sample['start_sec'] + sample['end_sec']) / 2) * fps)
mid = min(mid, len(vr) - 1)

frame = vr[mid].asnumpy()
print(f"FPS: {fps:.1f}  |  Total frames: {len(vr)}")

plt.figure(figsize=(7, 4))
plt.imshow(frame)
plt.title(f""{sample['caption']}"")
plt.axis('off')
plt.tight_layout()
plt.show()


## Done ✓
**What was produced:**
- `annotations/segments.csv` — full segment table
- `annotations/segments_available.csv` — filtered to downloaded videos
- `annotations/download_report.csv` — per-video status
- `videos/*.mp4` — raw video files
